# 固定闭合窗口的内部同步性
只比较同期市场描述；09:45/10:30预先固定，未来bar不能影响过去窗口，不称为预测信息。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.data.closed_intraday_window import closed_window_return
from src.data.qlib_native import load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/intraday_basket_observability_20261014';plan=json.loads((card/'plan.json').read_text())
for relative,digest in plan['provider_manifest_hashes'].items():assert hashlib.sha256((root/relative/'qlib_store_manifest.json').read_bytes()).hexdigest()==digest
qlib.init(provider_uri=str(root/plan['stock_provider']),region='us');stocks=load_qlib_feature_frame(QlibProviderConfig(root/plan['stock_provider'],freq='5min'),instruments=plan['symbols'],fields=['$open','$close','$volume','$price_valid'],start=plan['start'],end=plan['as_of']+' 16:00:00')
core=load_qlib_feature_frame(QlibProviderConfig(root/plan['core_provider'],freq='1min'),instruments=['SOXX'],fields=['$open','$close','$volume','$price_valid'],start=plan['start'],end=plan['as_of']+' 16:00:00')
def groups(native):
 assert native.attrs.get('qlib_data_api')==['qlib.data.D.features']
 x=native.reset_index().rename(columns={c:c.removeprefix('$') for c in native.columns});x['date']=x.datetime.dt.normalize();return dict(tuple(x.groupby(['instrument','date'])))
stock_groups,core_groups=groups(stocks),groups(core)
parent=root/plan['holdings_card'];s=json.loads((parent/'summary.json').read_text());mp=parent/'verification_manifest.json';assert hashlib.sha256(mp.read_bytes()).hexdigest()==plan['holdings_manifest_hash'];entry=next(e for e in json.loads(mp.read_text())['entries'] if e['path'].endswith('/artifacts/analysis/weights'))
with R.uri_context(s['recorder_uri']):
 rec=R.get_recorder(recorder_id=s['recorder_id'],experiment_name=parent.name);p=Path(rec.download_artifact('analysis/weights'));assert hashlib.sha256(p.read_bytes()).hexdigest()==entry['sha256'];weights=rec.load_object('analysis/weights')
rows=[]
for day,basket in weights.groupby('datetime'):
 assert pd.to_datetime(basket.filing_date).lt(day).all();w=basket.set_index('instrument').weight_pct;w=w/w.sum()
 for cutoff in plan['cutoffs']:
  returns=pd.Series({symbol:closed_window_return(stock_groups[(symbol,day)],day=day,cutoff=cutoff,frequency_minutes=5) for symbol in w.index});benchmark=closed_window_return(core_groups[('SOXX',day)],day=day,cutoff=cutoff,frequency_minutes=1);known=bool(returns.notna().all() and np.isfinite(benchmark))
  close_at=day+pd.Timedelta(hours=int(cutoff[:2]),minutes=int(cutoff[3:]))
  row={'date':day,'cutoff':cutoff,'window_closed_at':close_at,'assumed_available_at':close_at+pd.Timedelta(minutes=1),'known':known,'members':len(w),'soxx_return':benchmark,'positive_weight':np.nan,'weighted_stock_return':np.nan,'equal_weight_stock_return':np.nan,'cross_sectional_std':np.nan,'index_up_basket_not_majority':False,'index_down_basket_majority':False}
  if known:
   breadth=float(w.loc[returns.gt(0)].sum());weighted=float((w*returns).sum());row.update(positive_weight=breadth,weighted_stock_return=weighted,equal_weight_stock_return=float(returns.mean()),cross_sectional_std=float(returns.std(ddof=0)),index_up_basket_not_majority=bool(benchmark>0 and breadth<.5),index_down_basket_majority=bool(benchmark<0 and breadth>.5))
  rows.append(row)
observations=pd.DataFrame(rows);assert len(observations)==128
stats=[]
for cutoff,part in observations.groupby('cutoff'):
 known=part.loc[part.known];disagree=known.index_up_basket_not_majority|known.index_down_basket_majority
 stats.append({'cutoff':cutoff,'calendar_sessions':len(part),'known_sessions':len(known),'index_up_days':int(known.soxx_return.gt(0).sum()),'index_up_basket_not_majority':int(known.index_up_basket_not_majority.sum()),'index_down_basket_majority':int(known.index_down_basket_majority.sum()),'simultaneous_breadth_soxx_rank_correlation':float(known.positive_weight.corr(known.soxx_return,method='spearman')),'simultaneous_weighted_return_soxx_correlation':float(known.weighted_stock_return.corr(known.soxx_return)),'disagreement_abs_soxx_return_bp_median':float(known.loc[disagree,'soxx_return'].abs().median()*1e4)})
summary={'experiment':card.name,'decision':'closed_window_observability_only_no_leading_information_claim','new_fits':0,'new_backtests':0,'future_outcome_tests':0,'objective_achieved':False,'snapshots':len(observations),'known_snapshots':int(observations.known.sum()),'cutoffs':plan['cutoffs'],'historical_receive_time_certified':False,'discovery_window_reused':True,'provider_uri':str(root/plan['stock_provider']),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','scope':'Contemporaneous stock-return and ETF comparison, not predictive IC. Latest previously filed basket proxy, not current official index weights; cutoff+60s is a research availability assumption only.','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/snapshots','sources/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='fixed_closed_intraday_windows',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0)
 rec.save_objects(artifact_path='inputs',plan=plan,weights=weights)
 rec.save_objects(artifact_path='analysis',snapshots=observations,statistics=pd.DataFrame(stats),summary=summary)
 rec.save_objects(artifact_path='sources',window_contract=(root/'src/data/closed_intraday_window.py').read_text())
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),known_snapshots=int(observations.known.sum()))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(stats).to_csv(card/'window_observability.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':summary['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
for relative,digest in plan['provider_manifest_hashes'].items():assert hashlib.sha256((root/relative/'qlib_store_manifest.json').read_bytes()).hexdigest()==digest
print(pd.DataFrame(stats).to_string(index=False))


# 固定闭合窗口：可观察，但简单汇总与SOXX高度同动

本轮0拟合、0回测、0未来收益检验。固定09:45与10:30两个原有信息切点，在64个Q3交易日计算30成员披露篮子的同期描述，并与SOXX相同开盘到切点收益比较。实际成员权重必须在当日前已披露；并集不回填为每日真实指数成员。

两个切点共128个快照全部可计算。每只股票要求完整5分钟前缀、有效价格、首尾bar均有成交量，SOXX要求完整1分钟前缀。未来bar不进入当前切点；缺失前缀或首尾无新成交保持未知，不部分归一化。09:45用前三根5分钟bar，10:30用前十二根，不读取10:35收盘作为10:31信息。cutoff+60秒仅是历史研究可用时点假设，不是实时接收认证。

09:45成分加权收益与SOXX同期收益Pearson约0.9920，广度与SOXX同期收益Spearman约0.9505；10:30分别0.9971和0.9584。它们是同步性指标，不是预测IC。

09:45有5日SOXX上涨但上涨成分权重不足一半，2日SOXX下跌但多数权重上涨；共7日分歧，SOXX当时绝对涨跌幅中位11.87bp。10:30有4日第一类分歧、0日第二类，中位绝对涨跌幅仅3.34bp。保留全部日期，没有选择分歧日做收益或门槛搜索；接近零附近的方向变化不能自动视为强信号，披露权重陈旧和ETF成分偏差也可能影响比较。

决定：closed_window_observability_only_no_leading_information_claim。数据支持看见完整内部状态，但简单加权收益和上涨广度大部分反映共同市场走势；更多股票不自动提供更多领先信息。高同步也不证明所有内部结构均无增量。本卡不将这两个简单汇总加入模型，也不重开失败日频广度模型。

后续若研究新的日内机制，应事先说明其相对ETF同期走势的新增内容，并使用未用于提出机制的时间范围；Q3是已复用发现窗口，不能以这64日证明稳定超额。固定两个切点不因结果再调整。现役策略未改。

检查：未来bar不变性、缺失前缀、首尾零成交量合同测试通过，原生数据读取与源哈希核对、NAS回读通过。

[全部窗口统计](window_observability.csv) · [固定范围与来源](plan.json)

Recorder `9c1b78cadc99451f8728cd5874db78e5`。
